# Tutorial 7: Hand Tracking

## Introduction

In Aria-Gen2 glasses, one of the key upgrades from Aria-Gen1 is the capability to run Machine
Perception (MP) algorithms on the device during streaming / recording. Currently supported on-device
MP algorithms include Eye-tracking, Hand-tracking, and VIO. These algorithm results are stored as
separate data streams in the VRS file.

Hand tracking is produced by two independent sources, and this tutorial covers both:
the **on-device** stream inside the VRS file, and the **MPS** result computed in the
cloud after upload. They share the `HandTrackingResult` data structure, so the geometry
and drawing code below is written once and used for both.

| | On-device | MPS |
| :-- | :-- | :-- |
| Where | On the glasses, during recording | Cloud, after upload |
| Available | Immediately, and while streaming | After processing completes |
| Delivered as | `handtracking` VRS stream | `hand_tracking/hand_tracking_results.csv` |
| Read with | `VrsDataProvider.get_hand_pose_data_*` | `mps.hand_tracking.read_hand_tracking_results` or `MpsDataProvider` |

**What you'll learn:**

- How to access on-device HandTracking data from VRS files, and MPS hand tracking from CSV
- Understanding the concept of interpolated hand tracking and why interpolation is needed
- How to visualize HandTracking data projected onto 2D camera images using DeviceCalibration
- How to match MP data with camera frames using timestamps

**Where the other machine perception streams live**

| Signal | Tutorial |
| :-- | :-- |
| Eye gaze — on-device geometric, on-device ML, and MPS | `Tutorial_8_eyetracking` |
| Device pose — on-device VIO and MPS trajectory | `Tutorial_6_vio_and_trajectory` |
| MPS output layout and loading | `Tutorial_5_mps_basics` |

**Prerequisites**
- Complete Tutorial 1 (VrsDataProvider Basics) to understand basic data provider concepts
- Complete Tutorial 2 (Device Calibration) to understand how to properly use calibration in Aria data.
- Download Aria Gen2 sample data: [VRS](https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1.vrs) and [MPS output zip file](https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1_mps_output_dec_2025.zip)

### ⚠️ Important Notes
- **Google Colab Users:**  
  If you encounter a `ModuleNotFoundError: No module named 'rerun'` error after installing `rerun-sdk`, Colab may not recognize the new package until the runtime is restarted.  
  **Fix:** Go to **Runtime → Restart session and run all**.

- **Visualization Issue :**  
  If a Rerun visualization window does not appear, this may be due to a known caching issue. Simply re-run the visualization cell to resolve it.

## Setup Environment (Google Colab)

If running on Google Colab, install `projectaria-tools` and download sample data.


In [ ]:
import sys
import os
import subprocess

google_colab_env = 'google.colab' in str(get_ipython())

if google_colab_env:
    print("Running from Google Colab, installing projectaria_tools and downloading sample data")

    # Install projectaria-tools
    !pip install projectaria-tools==2.3.0

    # Set up data path
    vrs_sample_path = "./vrs_sample_data"

    # Sample VRS file and MPS output URLs
    vrs_url = "https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1.vrs"
    mps_url = "https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1_mps_output_dec_2025.zip"

    vrs_filename = "aria_gen2_sample_data_1.vrs"
    mps_zip_filename = "aria_gen2_sample_data_1_mps_output_dec_2025.zip"

    vrs_file_path = os.path.join(vrs_sample_path, vrs_filename)
    mps_zip_path = os.path.join(vrs_sample_path, mps_zip_filename)
    mps_folder_path = os.path.join(vrs_sample_path, "mps_output")

    # Download and unzip commands
    command_list = [
        f"mkdir -p {vrs_sample_path}",
        f'curl -o {vrs_file_path} -C - -O -L "{vrs_url}"',
        f'curl -o {mps_zip_path} -C - -O -L "{mps_url}"',
        f"unzip -o {mps_zip_path} -d {mps_folder_path}"
    ]

    # Execute the commands for downloading dataset
    print(f"Downloading VRS and MPS sample data...")
    for command in command_list:
        !$command

    print(f"Download complete! VRS file saved to: {vrs_file_path}")
    print(f"MPS data extracted to: {mps_folder_path}")

    # Running this command to trigger early failure of importing ReRun.
    # Should be resolved by restarting the Colab session.
    import rerun as rr
else:
    # For local environment, user needs to specify their own paths
    vrs_file_path = "path/to/your/recording.vrs"
    mps_folder_path = "path/to/your/mps/folder/"
    print(f"Please update vrs_file_path and mps_folder_path to point to your data")


In [ ]:
from projectaria_tools.core import data_provider

# Load VRS file
vrs_data_provider = data_provider.create_vrs_data_provider(vrs_file_path)


# Query HandTracking data streams
handtracking_label = "handtracking"
handtracking_stream_id = vrs_data_provider.get_stream_id_from_label(handtracking_label)
if handtracking_stream_id is None:
    raise RuntimeError(
        f"{handtracking_label} data stream does not exist! Please use a VRS that contains valid handtracking data for this tutorial."
    )

---

# Part A — On-device hand tracking

## On-Device Hand-tracking results
### Handtracking Data Structure
HandTracking data contains comprehensive 3D hand pose information. 
**Importantly, it directly reuses the [HandTrackingResults data structure](https://github.com/facebookresearch/projectaria_tools/blob/main/core/mps/HandTracking.h) from MPS (Machine Perception
Services)**, providing guaranteed compatibility across VRS and MPS.

**Key Fields in `HandTrackingResults`**
| Field Name           | Description                                                             |
| -------------------- | ----------------------------------------------------------------------- |
| `tracking_timestamp` | Timestamp of the hand-tracking estimate in the device time domain.      |
| `left_hand`          | Left-hand pose, or `None` if no valid pose is found for the timestamp.  |
| `right_hand`         | Right-hand pose, or `None` if no valid pose is found for the timestamp. |

**Single Hand fields (left or right):**
| Field Name                     | Description                                                                                                                                                                                                                          |
| ------------------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `confidence`                   | Tracking confidence score for this hand.                                                                                                                                                                                             |
| `landmark_positions_device`    | List of 21 hand-landmark positions in the device frame (3D points). <br>See the [wiki page](https://facebookresearch.github.io/projectaria_tools/docs/data_formats/mps/hand_tracking#hand_tracking_resultscsv) for landmark definitions. |
| `transform_device_wrist`       | Full SE3 transform of the wrist in the `Device` frame.                                                                                                                                                                               |
| `wrist_and_palm_normal_device` | Normal vectors for the wrist and palm joints in the `Device` frame.                                     

### Handtracking Coordinate System
All Handtracking results in Aria are stored in the `Device` coordinate frame, which is the same as device calibration. See `Tutorial_2_device_calibration` for definition of `Device` frame. 

In [ ]:
import numpy as np

def print_single_hand_information(single_hand):
    """
    A helper function to print the hand tracking result of one hand
    """
    print(f"\tConfidence: {single_hand.confidence:.3f}")
    print(
        f"\tLandmarks shape: {np.array(single_hand.landmark_positions_device).shape}"
    )
    print(
        f"\tWrist location: {single_hand.get_wrist_position_device()}"
    )
    print(
        f"\tPalm location: {single_hand.get_palm_position_device()}"
    )

print("=== HandTracking Data Sample ===")
num_handtracking_samples = vrs_data_provider.get_num_data(handtracking_stream_id)
selected_index = min(5, num_handtracking_samples)
hand_data = vrs_data_provider.get_hand_pose_data_by_index(
    handtracking_stream_id, selected_index
)

print(f"Sample {selected_index}:")
print(f"\tTracking timestamp: {hand_data.tracking_timestamp}")

# Print the content of left and right hand if valid
if hand_data.left_hand is not None:
    print("\tLeft hand detected")
    print_single_hand_information(hand_data.left_hand)
else:
    print("\tLeft hand: Not detected")

if hand_data.right_hand is not None:
    print("\tRight hand detected")
    print_single_hand_information(hand_data.right_hand)
else:
    print("\tRight hand: Not detected")

### Interpolated Hand-tracking Results
**Context:**

In Aria-Gen2 glasses, **the on-device hand-tracking data are calculated from the SLAM cameras, not RGB cameras**. 
In the mean time, the SLAM cameras and RGB camera often runs at different sampling frequency, and their triggering are not aligned either. 
This causes that the handtracking result's timestamp often do NOT line up with that of RGB camera, causing additional challenges in accurately visualize handtracking results in RGB images. 

**API to query interpolated handtracking results**

To resolve this, `vrs_data_provider` enables a special query API for handtracking results: 
```
vrs_data_provider.get_interpolated_hand_pose_data(stream_id, timestamp_ns)
```
which will return an interpolated handtracking results, given any timestamp within valid timestamps of the VRS file. 

**Handtracking Interpolation Implementation**

1. Find the 2 nearest hand-tracking results before and after the target timestamp.  
2. If the 2 hand-tracking results time delta is larger than 100 ms, interpolation is considered unreliable → return `None`.  
3. Otherwise, interpolate each hand separately:  
   a. For the left or right hand, perform interpolation **only if both the "before" and "after" samples contain a valid result for that hand**.  
   b. If either sample is missing, the interpolated result for that hand will be `None`.  Example:  
      ```text
      interpolate(
          before = [left = valid, right = None],
          after  = [left = valid, right = valid]
      )
      → result = [left = interpolated, right = None]
      ```
4. Single-hand interpolation is calculated as:  
   a. Apply linear interpolation on the 3D hand landmark positions.  
   b. Apply SE3 interpolation on `T_Device_Wrist` 3D pose.  
   c. Re-calculate the wrist and palm normal vectors.  
   d. Take the `min` of confidence values.  


In [ ]:
from projectaria_tools.core.sensor_data import SensorDataType, TimeDomain, TimeQueryOptions
from datetime import timedelta

print("\n=== Demonstrating query interpolated hand tracking results ===")

# Demonstrate how to query interpolated handtracking results
slam_stream_id = vrs_data_provider.get_stream_id_from_label("slam-front-left")
rgb_stream_id = vrs_data_provider.get_stream_id_from_label("camera-rgb")

# Retrieve a SLAM frame, use its timestamp as query
slam_sample_index = min(10, vrs_data_provider.get_num_data(slam_stream_id) - 1)
slam_data_and_record = vrs_data_provider.get_image_data_by_index(slam_stream_id, slam_sample_index)
slam_timestamp_ns = slam_data_and_record[1].capture_timestamp_ns

# Retrieve the closest RGB frame
rgb_data_and_record = vrs_data_provider.get_image_data_by_time_ns(
    rgb_stream_id, slam_timestamp_ns, TimeDomain.DEVICE_TIME, TimeQueryOptions.CLOSEST
)
rgb_timestamp_ns = rgb_data_and_record[1].capture_timestamp_ns

# Retrieve the closest hand tracking data sample
raw_ht_data = vrs_data_provider.get_hand_pose_data_by_time_ns(
    handtracking_stream_id, slam_timestamp_ns, TimeDomain.DEVICE_TIME, TimeQueryOptions.CLOSEST
)
raw_ht_timestamp_ns = (raw_ht_data.tracking_timestamp // timedelta(microseconds=1)) * 1000

# Check if hand tracking aligns with RGB or SLAM data
print(f"SLAM timestamp: {slam_timestamp_ns}")
print(f"RGB timestamp:  {rgb_timestamp_ns}")
print(f"hand tracking timestamp:   {raw_ht_timestamp_ns}")
print(f"hand tracking-SLAM time diff: {abs(raw_ht_timestamp_ns - slam_timestamp_ns) / 1e6:.2f} ms")
print(f"hand tracking- RGB time diff: {abs(raw_ht_timestamp_ns - rgb_timestamp_ns) / 1e6:.2f} ms")

# Now, query interpolated hand tracking data sample using RGB timestamp.
interpolated_ht_data = vrs_data_provider.get_interpolated_hand_pose_data(
    handtracking_stream_id, rgb_timestamp_ns
)

# Check that interpolated hand tracking now aligns with RGB data
if interpolated_ht_data is not None:
    interpolated_ht_timestamp_ns = (interpolated_ht_data.tracking_timestamp// timedelta(microseconds=1)) * 1000
    print(f"Interpolated hand tracking timestamp: {interpolated_ht_timestamp_ns}")
    print(f"Interpolated hand tracking-RGB time diff: {abs(interpolated_ht_timestamp_ns - rgb_timestamp_ns) / 1e6:.2f} ms")
else:
    print("Interpolated hand tracking data is None - interpolation failed")

### Visualize Hand-tracking Results in Cameras
In this section, we show some example code on how to visualize the hand-tracking results in SLAM and RGB camera images. 
Basically, you need to project the hand tracking results (landmarks, skeleton lines) into the camera images using the camera's calibration. 

In [ ]:
import rerun as rr
from projectaria_tools.core.sensor_data import SensorDataType, TimeDomain, TimeQueryOptions
from projectaria_tools.utils.rerun_helpers import create_hand_skeleton_from_landmarks

def plot_single_hand_in_camera(hand_joints_in_device, camera_label, camera_calib, hand_label):
    """
    A helper function to plot a single hand data in 2D camera view
    """
    # Setting different marker plot sizes for RGB and SLAM since they have different resolutions
    plot_ratio = 3.0 if camera_label == "camera-rgb" else 1.0
    marker_color = [255,64,0] if hand_label == "left" else [255, 255, 0]

    # project into camera frame, and also create line segments
    hand_joints_in_camera = []
    for pt_in_device in hand_joints_in_device:
        pt_in_camera = (
            camera_calib.get_transform_device_camera().inverse() @ pt_in_device
        )
        pixel = camera_calib.project(pt_in_camera)
        hand_joints_in_camera.append(pixel)

    # Create hand skeleton in 2D image space
    hand_skeleton = create_hand_skeleton_from_landmarks(hand_joints_in_camera)

    # Remove "None" markers from hand joints in camera. This is intentionally done AFTER the hand skeleton creation
    hand_joints_in_camera = list(
        filter(lambda x: x is not None, hand_joints_in_camera)
    )

    rr.log(
        f"{camera_label}/{hand_label}/landmarks",
        rr.Points2D(
            positions=hand_joints_in_camera,
            colors= marker_color,
            radii= [3.0 * plot_ratio]
        ),
    )
    rr.log(
        f"{camera_label}/{hand_label}/skeleton",
        rr.LineStrips2D(
            hand_skeleton,
            colors=[0, 255, 0],
            radii= [0.5 * plot_ratio],
        ),
    )

def plot_handpose_in_camera(hand_pose, camera_label, camera_calib):
    """
    A helper function to plot hand tracking results into a camera image
    """
    # Clear the canvas first
    #rr.log(
    #    f"{camera_label}/handtracking",
    #    rr.Clear.recursive(),
    #)

    # Plot both hands
    if hand_pose.left_hand is not None:
        plot_single_hand_in_camera(
            hand_joints_in_device=hand_pose.left_hand.landmark_positions_device,
            camera_label=camera_label,
            camera_calib = camera_calib,
            hand_label="left")
    if hand_pose.right_hand is not None:
        plot_single_hand_in_camera(
            hand_joints_in_device=hand_pose.right_hand.landmark_positions_device,
            camera_label=camera_label,
            camera_calib = camera_calib,
            hand_label="right")


In [ ]:
print("\n=== Visualizing on-device hand tracking in camera images ===")

# First, query the RGB camera stream id
device_calib = vrs_data_provider.get_device_calibration()
rgb_camera_label = "camera-rgb"
slam_camera_labels = ["slam-front-left", "slam-front-right", "slam-side-left", "slam-side-right"]
rgb_stream_id = vrs_data_provider.get_stream_id_from_label(rgb_camera_label)
slam_stream_ids = [vrs_data_provider.get_stream_id_from_label(label) for label in slam_camera_labels]

rr.init("rerun_viz_ht_in_cameras")

# Set up a sensor queue with only RGB images.
# Handtracking data will be queried with interpolated API.
deliver_options = vrs_data_provider.get_default_deliver_queued_options()
deliver_options.deactivate_stream_all()
for stream_id in slam_stream_ids + [rgb_stream_id]:
    deliver_options.activate_stream(stream_id)

# Play for only 3 seconds
total_length_ns = vrs_data_provider.get_last_time_ns_all_streams(TimeDomain.DEVICE_TIME) - vrs_data_provider.get_first_time_ns_all_streams(TimeDomain.DEVICE_TIME)
skip_begin_ns = int(15 * 1e9) # Skip 15 seconds
duration_ns = int(3 * 1e9) # 3 seconds
skip_end_ns = max(total_length_ns - skip_begin_ns - duration_ns, 0)
deliver_options.set_truncate_first_device_time_ns(skip_begin_ns)
deliver_options.set_truncate_last_device_time_ns(skip_end_ns)

# Plot image data, and overlay hand tracking data
for sensor_data in vrs_data_provider.deliver_queued_sensor_data(deliver_options):
    # ---------------
    # Only image data will be obtained.
    # ---------------
    device_time_ns = sensor_data.get_time_ns(TimeDomain.DEVICE_TIME)
    image_data_and_record = sensor_data.image_data_and_record()
    stream_id = sensor_data.stream_id()
    camera_label = vrs_data_provider.get_label_from_stream_id(stream_id)
    camera_calib = device_calib.get_camera_calib(camera_label)


    # Visualize the RGB images.
    rr.set_time("device_time", duration=np.timedelta64(device_time_ns, "ns"))
    rr.log(f"{camera_label}", rr.Image(image_data_and_record[0].to_numpy_array()))

    # Query and plot interpolated hand tracking result
    interpolated_hand_pose = vrs_data_provider.get_interpolated_hand_pose_data(handtracking_stream_id, device_time_ns, TimeDomain.DEVICE_TIME)
    if interpolated_hand_pose is not None:
        plot_handpose_in_camera(hand_pose = interpolated_hand_pose, camera_label = camera_label, camera_calib = camera_calib)

# Wait for rerun to buffer 1 second of data
import time
time.sleep(1)

rr.notebook_show()

---

# Part B — MPS hand tracking

Everything above came off the glasses in real time. MPS re-runs hand tracking in the
cloud from the same recording, without the compute and power limits of the device, and
writes the result next to the other MPS outputs. `Tutorial_5_mps_basics` covers the
folder layout; this part goes straight to the hand tracking output.

## Reading the MPS result

The MPS Hand Tracking algorithm augments each sequence with temporal hand pose estimates produced offline. Each sequence folder contains a `hand_tracking/` directory with `hand_tracking_results.csv` and metadata summaries that describe the quality of the run. The CSV stores one `HandTrackingResult` per device timestamp and includes left/right hand outputs when a hand is detected.

#### Hand Tracking Outputs
- **21 landmarks** per detected hand expressed in the device frame
- **Wrist-to-device transform** capturing the full 6DoF pose of the wrist
- **Palm and wrist normals** to reason about hand orientation
- **Confidence scores** indicating tracking quality for each hand

#### Query Utilities
The `projectaria_tools.core.mps.hand_tracking` module exposes helpers for loading and working with results:
- `read_hand_tracking_results(path)` loads the full time series into memory
- `HandTrackingResult` objects provide direct access to landmark positions, wrist transforms, and helper methods such as `get_wrist_position_device()`

Note that these are the same fields the on-device stream carries in Part A -- the two sources share the `HandTrackingResult` type, which is what lets the drawing code below be reused unchanged.

In [ ]:
import os

from projectaria_tools.core import mps

print("=== MPS - Hand Tracking ===")

hand_tracking_results_file = os.path.join(
    mps_folder_path, "hand_tracking", "hand_tracking_results.csv"
)
hand_tracking_results = mps.hand_tracking.read_hand_tracking_results(
    hand_tracking_results_file
)

if hand_tracking_results:
    if len(hand_tracking_results) > 10:
        sample = hand_tracking_results[10] # get stable hand tracking result, since first hand tracking result in example vrs might be empty
    else:
        sample = hand_tracking_results[0]
    sample_ts_us = int(sample.tracking_timestamp.total_seconds() * 1e6)
    print(f"Sample tracking timestamp: {sample_ts_us} us")
    print(f"Total number of hand tracking results: {len(hand_tracking_results)}")

    for handedness, hand in (("Left", sample.left_hand), ("Right", sample.right_hand)):
        if hand is None:
            print(f"  {handedness} hand: not available in this sample")
            continue

        landmarks = hand.landmark_positions_device
        landmark_count = len(landmarks) if landmarks is not None else 0
        print(f"  {handedness} hand confidence: {hand.confidence:.2f}")
        print(f"  {handedness} hand landmark count: {landmark_count}")
        wrist_position = hand.get_wrist_position_device()
        palm_position = hand.get_palm_position_device()
        print(
            f"  {handedness} wrist position (device frame): {wrist_position}"
        )
        print(
            f"  {handedness} palm position (device frame): {palm_position}"
        )
else:
    print("hand_tracking_results is empty.")

## Querying MPS hands by timestamp

Reading the whole CSV is fine for analysis over the full sequence, but to line hands up
against camera frames you want a timestamp query. `MpsDataProvider` gives you the same
two access patterns the VRS side has in Part A, including an interpolating one:

| On-device (Part A) | MPS |
| :-- | :-- |
| `get_hand_pose_data_by_time_ns(...)` | `MpsDataProvider.get_hand_tracking_result(...)` |
| `get_interpolated_hand_pose_data(...)` | `MpsDataProvider.get_interpolated_hand_tracking_result(...)` |

**The two sources are stamped against different frames, so do not carry Part A's
assumption over.** Measured on the sample recording:

| | Lands on `slam-front-left` frames | Lands on `camera-rgb` frames |
| :-- | :-- | :-- |
| On-device hand tracking | to 0.001 ms | 0.93 ms median, 4.33 ms worst |
| MPS hand tracking | 0.001 ms median, 3.25 ms worst | to 0.001 ms |

On-device hand tracking is stamped at SLAM camera times and has to be interpolated onto
an RGB frame -- that is what Part A's interpolated query is for. MPS hand tracking on
this recording already lands exactly on RGB frame times, at 30 Hz, so a plain `CLOSEST`
query returns the right sample and interpolating changes nothing.

That leaves `get_interpolated_hand_tracking_result` for the *other* direction: a query
time that is not on the output's own grid, such as a SLAM frame time, an IMU sample, or
a fixed rate of your own. Measure the offset on your own data rather than assuming
either layout -- the cell below prints it for both sources.

In [ ]:
mps_data_paths = mps.MpsDataPathsProvider(mps_folder_path).get_data_paths()
mps_data_provider = mps.MpsDataProvider(mps_data_paths)

print(f"has_hand_tracking_results(): {mps_data_provider.has_hand_tracking_results()}")
print(f"get_hand_tracking_version():  {mps_data_provider.get_hand_tracking_version()}")

if mps_data_provider.has_hand_tracking_results():
    # Use an RGB frame time, the same query the visualization below makes.
    rgb_stream_id = vrs_data_provider.get_stream_id_from_label("camera-rgb")
    probe_index = min(30, vrs_data_provider.get_num_data(rgb_stream_id) - 1)
    probe_time_ns = vrs_data_provider.get_image_data_by_index(
        rgb_stream_id, probe_index
    )[1].capture_timestamp_ns

    nearest = mps_data_provider.get_hand_tracking_result(probe_time_ns)
    interpolated = mps_data_provider.get_interpolated_hand_tracking_result(probe_time_ns)

    print(f"\nRGB frame at {probe_time_ns} ns")
    for name, result in (("nearest", nearest), ("interpolated", interpolated)):
        if result is None:
            print(f"  {name}: None")
            continue
        result_ns = int(result.tracking_timestamp.total_seconds() * 1e9)
        print(f"  {name:12} tracking_timestamp {result_ns} ns "
              f"({(result_ns - probe_time_ns) / 1e6:+.3f} ms from the frame)")

    # The same question asked of the on-device stream, so the two are comparable.
    on_device = vrs_data_provider.get_hand_pose_data_by_time_ns(
        handtracking_stream_id, probe_time_ns, TimeDomain.DEVICE_TIME, TimeQueryOptions.CLOSEST
    )
    on_device_ns = int(on_device.tracking_timestamp.total_seconds() * 1e9)
    print(f"  {'on-device':12} tracking_timestamp {on_device_ns} ns "
          f"({(on_device_ns - probe_time_ns) / 1e6:+.3f} ms from the frame)")

## Drawing MPS hands on the RGB image

The projection is identical to Part A, and so is the code: `plot_handpose_in_camera`
takes anything exposing `left_hand` / `right_hand` with `landmark_positions_device`, and
both sources satisfy that. Only the query changes.

Drawing both sources on the same frame is the useful thing to do here -- it is the only
way to see where they disagree.

In [ ]:
print("\n=== Visualizing on-device and MPS hand tracking on RGB images ===")

if not mps_data_provider.has_hand_tracking_results():
    print("This MPS output has no hand tracking results - skipping.")
else:
    rgb_camera_label = "camera-rgb"
    rgb_stream_id = vrs_data_provider.get_stream_id_from_label(rgb_camera_label)
    device_calib = vrs_data_provider.get_device_calibration()
    rgb_camera_calib = device_calib.get_camera_calib(rgb_camera_label)

    rr.init("rerun_viz_ht_on_device_vs_mps")

    deliver_options = vrs_data_provider.get_default_deliver_queued_options()
    deliver_options.deactivate_stream_all()
    deliver_options.activate_stream(rgb_stream_id)

    # Same 3 second window as Part A, so the two visualizations are comparable.
    total_length_ns = vrs_data_provider.get_last_time_ns_all_streams(
        TimeDomain.DEVICE_TIME
    ) - vrs_data_provider.get_first_time_ns_all_streams(TimeDomain.DEVICE_TIME)
    skip_begin_ns = int(15 * 1e9)
    duration_ns = int(3 * 1e9)
    deliver_options.set_truncate_first_device_time_ns(skip_begin_ns)
    deliver_options.set_truncate_last_device_time_ns(
        max(total_length_ns - skip_begin_ns - duration_ns, 0)
    )

    for sensor_data in vrs_data_provider.deliver_queued_sensor_data(deliver_options):
        device_time_ns = sensor_data.get_time_ns(TimeDomain.DEVICE_TIME)
        image_data_and_record = sensor_data.image_data_and_record()

        rr.set_time("device_time", duration=np.timedelta64(device_time_ns, "ns"))
        rr.log(rgb_camera_label, rr.Image(image_data_and_record[0].to_numpy_array()))

        on_device_pose = vrs_data_provider.get_interpolated_hand_pose_data(
            handtracking_stream_id, device_time_ns, TimeDomain.DEVICE_TIME
        )
        mps_pose = mps_data_provider.get_interpolated_hand_tracking_result(device_time_ns)

        # Two entity roots so Rerun draws them as separate, individually toggleable
        # overlays rather than one indistinguishable pile of landmarks.
        for root, pose in ((f"{rgb_camera_label}/on_device", on_device_pose),
                           (f"{rgb_camera_label}/mps", mps_pose)):
            if pose is None:
                rr.log(root, rr.Clear.recursive())
                continue
            plot_handpose_in_camera(
                hand_pose=pose, camera_label=root, camera_calib=rgb_camera_calib
            )

    import time
    time.sleep(1)
    rr.notebook_show()

---

## Related tutorials

- `Tutorial_1_vrs_data_provider_basics` — the query APIs used throughout
- `Tutorial_2_device_calibration` — the `Device` frame and camera projection used to draw the hands
- `Tutorial_5_mps_basics` — MPS output layout, `MpsDataPathsProvider` and `MpsDataProvider`
- `Tutorial_6_vio_and_trajectory` — device pose, the other algorithm with an on-device and an MPS source
- `Tutorial_8_eyetracking` — eye gaze, which has three sources rather than two